In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0]))

import numpy as np
import pandas as pd

from config import PROCESSED_DATA_DIR

In [2]:
pokemon_df = pd.read_parquet(PROCESSED_DATA_DIR / "pokemon_data_clean.parquet")

In [3]:
print(pokemon_df.columns)

Index(['name', 'generation', 'status', 'type_number', 'type_1', 'type_2',
       'height_m', 'weight_kg', 'abilities_number', 'total_points',
       'catch_rate', 'base_friendship', 'base_experience', 'growth_rate',
       'against_normal', 'against_fire', 'against_water', 'against_electric',
       'against_grass', 'against_ice', 'against_fight', 'against_poison',
       'against_ground', 'against_flying', 'against_psychic', 'against_bug',
       'against_rock', 'against_ghost', 'against_dragon', 'against_dark',
       'against_steel', 'against_fairy', 'ability_1', 'ability_2',
       'ability_hidden'],
      dtype='str')


In [4]:
# Log-transform weight for a cleaner linear-model input than raw weight.
pokemon_df["log_weight_kg"] = np.log1p(pokemon_df["weight_kg"])

In [5]:
# Also log-transform height for cleaner linear-model input 
pokemon_df["log_height_m"] = np.log1p(pokemon_df["height_m"])

In [6]:
# Add features for special variants that may not be captured by the baseline structure alone.
pokemon_df["is_mega"] = pokemon_df["name"].str.contains("mega", case=False, na=False).astype(int)
pokemon_df["is_primal"] = pokemon_df["name"].str.contains("primal", case=False, na=False).astype(int)
pokemon_df["is_form"] = pokemon_df["name"].str.contains("form", case=False, na=False).astype(int)

In [7]:
# Keep a lean baseline feature set plus the prediction target.
pokemon_df = pokemon_df[[
    "name",
    "total_points",
    "generation",
    "log_height_m",
    "log_weight_kg",
    "is_mega",
    "is_primal",
    "is_form",
    "status",
    "type_1",
    "type_2",
    "growth_rate",
]]

In [8]:
categorical_cols = ["status", "type_1", "type_2", "growth_rate"]

In [9]:
pokemon_df = pd.get_dummies(
    pokemon_df,
    columns=categorical_cols,
    drop_first=True,
)

In [10]:
print(pokemon_df.shape)
print(pokemon_df.columns.tolist())

(1045, 51)
['name', 'total_points', 'generation', 'log_height_m', 'log_weight_kg', 'is_mega', 'is_primal', 'is_form', 'status_mythical', 'status_normal', 'status_sub legendary', 'type_1_dark', 'type_1_dragon', 'type_1_electric', 'type_1_fairy', 'type_1_fighting', 'type_1_fire', 'type_1_flying', 'type_1_ghost', 'type_1_grass', 'type_1_ground', 'type_1_ice', 'type_1_normal', 'type_1_poison', 'type_1_psychic', 'type_1_rock', 'type_1_steel', 'type_1_water', 'type_2_dark', 'type_2_dragon', 'type_2_electric', 'type_2_fairy', 'type_2_fighting', 'type_2_fire', 'type_2_flying', 'type_2_ghost', 'type_2_grass', 'type_2_ground', 'type_2_ice', 'type_2_none', 'type_2_normal', 'type_2_poison', 'type_2_psychic', 'type_2_rock', 'type_2_steel', 'type_2_water', 'growth_rate_fast', 'growth_rate_fluctuating', 'growth_rate_medium fast', 'growth_rate_medium slow', 'growth_rate_slow']


In [11]:
pokemon_df.to_parquet(PROCESSED_DATA_DIR / "pokemon_data_features.parquet", index=False)